# Red Five — independent quantile diagnostics

Synthetic data only. Fit fixed reference bins before an explicit training cutoff, then evaluate later scores without refitting. Each time-series instrument/contract is its own model. No residualization, weight fitting, significance or acceptance verdict is produced.

In [ ]:
%matplotlib inline
from pathlib import Path

from IPython.display import display
from matplotlib.figure import Figure

from red_five.component_export import Component, export_components
from red_five.composition import PlotOptions, Selection
from red_five.quantiles import QuantileConfig
from red_five.rendering import verify_bundle
from red_five.sections import evaluate_section

ROOT = next(
    p
    for p in (Path.cwd(), *Path.cwd().parents)
    if (p / "examples/quantile-evaluation.json").is_file()
)
result = evaluate_section(
    "quantiles",
    (ROOT / "examples/quantile-signals.csv").read_bytes(),
    (ROOT / "examples/quantile-evaluation.json").read_bytes(),
    (ROOT / "STATISTICAL_ANALYSIS_PLAN.md").read_bytes(),
    (ROOT / "uv.lock").read_bytes(),
    quantiles=QuantileConfig(
        training_end="2026-01-21T00:00:00Z",
        bins=5,
        minimum_training=20,
        minimum_bin=3,
    ),
)
print(result.section_id)
display(result.diagnostics)

## Inspect each portion separately

Training uses score observations strictly before the cutoff, never training returns. Bin equality goes to the lower bin; repeated boundaries collapse. Evaluation scores outside the training range enter tail bins and are counted. Unavailable means remain null, not zero. Counts describe coverage, not independent sample size.

In [ ]:
es = result.select(Selection(model_ids=("ES-model",), precision=4))
display(es)
display(es.table())
display(es.figure(options=PlotOptions(title="ES forward return by frozen score bin")))

In [ ]:
display(es.figure("quantile_counts", options=PlotOptions(title="ES label coverage")))

## Group-level spread and monotonicity table

Spread is highest-bin minus lowest-bin mean return, **not** a traded long/short portfolio return. These group-level values repeat on the tidy bin rows; display once per group, never sum the repeats. Monotonicity is descriptive and requires all effective-bin means.

In [ ]:
group_summary = (
    result.select()
    .table()
    .select(
        "model_id",
        "instrument_id",
        "contract_id",
        "spread",
        "spread_reason",
        "monotonicity",
        "monotonicity_reason",
    )
    .unique(maintain_order=True)
)
display(group_summary)

## Your layout and styling

Use separate axes for unrelated model scales. Customize colors, size, titles and limits without recalculating evidence. You can also plot the returned Polars table yourself; custom manual edits are not replayed by the standard bundle exporter.

In [ ]:
figure = Figure(figsize=(11, 9), layout="constrained")
axes = figure.subplots(2, 1)
for ax, model, color in zip(
    axes,
    ("ES-model", "NQ-model"),
    ("#0072B2", "#D55E00"),
    strict=True,
):
    result.select(Selection(model_ids=(model,))).plot(
        ax=ax,
        options=PlotOptions(title=f"{model}: frozen-bin means", colors=(color,)),
    )
display(figure)

## Export only selected components

The bundle contains stored section evidence, selected exact-value tables, charts and a verified partial manifest. No full-report evaluation occurs. A content-based folder avoids overwriting different evidence. Selection below changes presentation only, not the fitted boundaries or sample.

In [ ]:
output = ROOT / "build" / f"quantile-components-{result.section_id[:16]}"
export_components(
    [
        Component(es, "quantiles", PlotOptions(title="ES bin means")),
        Component(es, "quantile_counts", PlotOptions(title="ES coverage")),
        Component(result.select(Selection(model_ids=("NQ-model",)))),
    ],
    output,
)
assert verify_bundle(output)["scope"] == "partial"
print(output)

## Limits and next evidence

These are fixed training-reference bins, not per-date equal-count ranks. Cross-sectional mode fits per-model historical panel scores and reports each evaluation date separately; comparable scores within each model are an upstream requirement. One declared cutoff is not a purged walk-forward study. Overlapping labels, upstream training leakage, multiplicity and uncertainty remain unresolved. Do not tune bins/cutoffs on these results and treat them as confirmatory. See `docs/QUANTILES.md`.